In [1]:
import joblib

In [9]:
from pathlib import Path
import pandas as pd
import numpy as np
import tarfile
import urllib.request

In [10]:
def load_housing_data():
    tarball_path = Path("../datasets/housing.tgz")
    if not tarball_path.is_file():
        Path("../datasets").mkdir(parents=True,exist_ok=True)
        url = "https://github.com/ageron/data/raw/main/housing.tgz"
        urllib.request.urlretrieve(url, tarball_path)
    with tarfile.open(tarball_path) as f:
        f.extractall(path="../datasets")
    return pd.read_csv(Path("../datasets/housing/housing.csv"))
housing = load_housing_data()

In [12]:
new_data = housing.iloc[:5]

In [3]:
from sklearn.cluster import KMeans
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.metrics.pairwise import rbf_kernel

In [6]:
def column_ratio(X):
    return X[:,[0]] / X[:,[1]]
    
def ratio_name(function_transformer, feature_names_in):
    return ["ratio"] # feature names out
    
def ratio_pipeline():
    return make_pipeline(
        SimpleImputer(strategy='median'),
        FunctionTransformer(column_ratio, feature_names_out=ratio_name),
        StandardScaler()
    )
class ClusterSimilarity(BaseEstimator, TransformerMixin):
    def __init__(self,n_clusters=10, gamma=1.0, random_state=None):
        self.n_clusters = n_clusters
        self.gamma = gamma
        self.random_state = random_state
        
    def fit(self, X, y=None, sample_weight=None):
        self.kmeans_ = KMeans(self.n_clusters,n_init=10, random_state= self.random_state)
        self.kmeans_.fit(X, sample_weight=sample_weight)
        return self # always return self
        
    def transform(self, X):
        return rbf_kernel(X, self.kmeans_.cluster_centers_, gamma=self.gamma)
        
    def get_feature_names_out(self, names = None):
        return [f"Cluster {i} similarity" for i in range(self.n_clusters)]

In [7]:
final_model_reloaded = joblib.load("my_california_housing_model.pkl")

In [13]:
final_model_reloaded.predict(new_data)

array([452600., 358500., 352100., 341300., 342200.])